# Machine Learning Modeling

In [1]:
# imports
import pandas as pd 
import numpy as np 
import json
import joblib
import warnings
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report,
                             confusion_matrix,
                             accuracy_score,
                             f1_score)
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

print("Imports done ")

Imports done 


# Model A -- Visit Risk Classification
## Business Purpose:
Predict wether a hospital visit represents Low,Medium, or High operational and clinical risk.

In [2]:
# load the dataset
df = pd.read_csv("../outputs/model_table.csv",parse_dates=["registration_date","visit_date","billing_date"])
print("Shape: ",df.shape)
df.head()

Shape:  (25000, 30)


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,...,risk_numeric,claim_numeric,is_rejected,days_since_registration,visit_frequency,avg_los_per_patient,provider_rejection_rate,visit_month,visit_dayofweek,high_cost_visit_flag
0,2,15,F,Mumbai,CareOne,0,2025-12-27,8,2026-01-01,General,...,NaN,2,1,5,4,21.120000,0.256876,1,3,False
1,12,3,M,Bangalore,CareOne,0,2025-08-13,65,2026-01-01,ICU,...,2.0,2,1,141,8,23.750000,0.256876,1,3,True
2,129,44,M,Pune,MediCareX,1,2025-07-20,651,2026-01-01,ICU,...,2.0,1,0,165,3,32.460000,0.242556,1,3,True
3,133,47,F,Delhi,CareOne,1,2025-11-02,670,2026-01-01,General,...,NaN,0,0,60,3,30.056667,0.256876,1,3,False
4,139,14,F,Chennai,SecureLife,1,2025-02-05,706,2026-01-01,Cardiology,...,NaN,0,0,330,9,29.030000,0.157496,1,3,True


In [3]:
# risk fea\tures
risk_target = "risk_score"
risk_features = [
    "age",
    "gender",
    "city",
    "insurance_provider",
    "chronic_flag",
    "department",
    "visit_type",
    "doctor_id",
    "length_of_stay_hours",
    "days_since_registration",
    "visit_frequency",
    "avg_los_per_patient",
    "visit_month",
    "visit_dayofweek"
]
risk_df = df[risk_features + [risk_target, "visit_date"]].copy()
risk_df = risk_df.dropna(subset=[risk_target,"visit_date"])
print("Risk dataset shape:", risk_df.shape)
risk_df.head()

Risk dataset shape: (25000, 16)


,age,gender,city,insurance_provider,chronic_flag,department,visit_type,doctor_id,length_of_stay_hours,days_since_registration,visit_frequency,avg_los_per_patient,visit_month,visit_dayofweek,risk_score,visit_date
0,15,F,Mumbai,CareOne,0,General,OPD,105,9.63,5,4,21.120000,1,3,Low,2026-01-01
1,3,M,Bangalore,CareOne,0,ICU,ICU,112,59.60,141,8,23.750000,1,3,High,2026-01-01
2,44,M,Pune,MediCareX,1,ICU,ER,150,59.28,165,3,32.460000,1,3,High,2026-01-01
3,47,F,Delhi,CareOne,1,General,OPD,145,25.15,60,3,30.056667,1,3,Medium,2026-01-01
4,14,F,Chennai,SecureLife,1,Cardiology,ER,148,42.88,330,9,29.030000,1,3,Medium,2026-01-01


In [4]:
# Time based split
risk_df = risk_df.sort_values("visit_date").reset_index(drop=True)
split_idx = int(len(risk_df)*0.8)

risk_train = risk_df.iloc[:split_idx].copy()
risk_test = risk_df.iloc[split_idx:].copy()

X_train_risk = risk_train[risk_features]
X_test_risk = risk_test[risk_features]

y_train_risk = risk_train[risk_target]
y_test_risk = risk_test[risk_target]

print("Train shape: ",X_train_risk.shape)
print("Test shape: ",X_test_risk.shape)
print("Train Period: ",risk_train["visit_date"].min().date(),"->", risk_train["visit_date"].max().date())
print("Test Period: ",risk_test["visit_date"].min().date(),"->", risk_test["visit_date"].max().date())

Train shape:  (20000, 14)
Test shape:  (5000, 14)
Train Period:  2025-01-21 -> 2026-01-02
Test Period:  2026-01-02 -> 2026-01-20


## Preprocessing Pipeline
**Two transformers:**
- Numeric -> SimpleImputer (median strategy)
- Categorical -> SimpleImputer -> OneHotEncoder


Combined using ColumnTransformer. this entire pipeline gets saved with the model -  no seperate preprocessing step at inference time.

In [5]:
# Preprocessing pipeline
risk_numeric_features = [
    "age",
    "chronic_flag",
    "length_of_stay_hours",
    "days_since_registration",
    "visit_frequency",
    "doctor_id",
    "avg_los_per_patient",
    "visit_month",
    "visit_dayofweek"
]

risk_categorical_features = [
    "gender",
    "city",
    "insurance_provider",
    "department",
    "visit_type"
]

risk_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

risk_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

risk_preprocessor = ColumnTransformer(transformers=[
    ("num", risk_numeric_transformer, risk_numeric_features),
    ("cat", risk_categorical_transformer, risk_categorical_features)
])

print("Preprocessing pipeline ready")

Preprocessing pipeline ready


## Baseline model: Logistic Regression

In [6]:
# Logistic Regression Model
risk_baseline_model = Pipeline(steps=[
    ("preprocessor", risk_preprocessor),
    ("classifier", LogisticRegression(
        max_iter = 1000,
        class_weight="balanced"
    ))
])

# Train
risk_baseline_model.fit(X_train_risk, y_train_risk)

# Predict
risk_baseline_pred_train = risk_baseline_model.predict(X_train_risk)
risk_baseline_pred_test =  risk_baseline_model.predict(X_test_risk)

baseline_acc = accuracy_score(y_test_risk, risk_baseline_pred_test)
print("Risk Baseline Train Accuracy :", round(accuracy_score(y_train_risk, risk_baseline_pred_train), 4))
print("Risk Baseline Test Accuracy  :", round(accuracy_score(y_test_risk,  risk_baseline_pred_test), 4))
print("Risk Baseline Test Weighted F1:", round(f1_score(y_test_risk, risk_baseline_pred_test, average="weighted"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test_risk, risk_baseline_pred_test))
print("Confusion Matrix:\n", confusion_matrix(y_test_risk, risk_baseline_pred_test))

c:\Users\ayata\OneDrive\Healthcare ml project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Risk Baseline Train Accuracy : 0.9218
Risk Baseline Test Accuracy  : 0.9208
Risk Baseline Test Weighted F1: 0.9211

Classification Report:

              precision    recall  f1-score   support

        High       0.90      0.95      0.93       838
         Low       0.96      0.93      0.94      2360
      Medium       0.88      0.90      0.89      1802

    accuracy                           0.92      5000
   macro avg       0.91      0.93      0.92      5000
weighted avg       0.92      0.92      0.92      5000

Confusion Matrix:
 [[ 796    0   42]
 [   0 2186  174]
 [  86   94 1622]]


## Random Forest Model

In [10]:
# Random Forest Model
risk_rf_model = Pipeline(steps=[
    ("preprocessor", risk_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_split=20,
        min_samples_leaf=10,
        class_weight = "balanced_subsample",
        random_state=42
    ))
])

# Train
risk_rf_model.fit(X_train_risk, y_train_risk)

# Predict
risk_rf_pred_train = risk_rf_model.predict(X_train_risk)
risk_rf_pred_test =  risk_rf_model.predict(X_test_risk)

rf_acc = accuracy_score(y_test_risk, risk_rf_pred_test)
print("Risk Baseline Train Accuracy :", round(accuracy_score(y_train_risk, risk_rf_pred_train), 4))
print("Risk Baseline Test Accuracy  :", round(accuracy_score(y_test_risk,  risk_rf_pred_test), 4))
print("Risk Baseline Test Weighted F1:", round(f1_score(y_test_risk, risk_rf_pred_test, average="weighted"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test_risk, risk_rf_pred_test))
print("Confusion Matrix:\n", confusion_matrix(y_test_risk, risk_rf_pred_test))

Risk Baseline Train Accuracy : 0.9414
Risk Baseline Test Accuracy  : 0.935
Risk Baseline Test Weighted F1: 0.935

Classification Report:

              precision    recall  f1-score   support

        High       0.95      0.93      0.94       838
         Low       0.95      0.95      0.95      2360
      Medium       0.91      0.91      0.91      1802

    accuracy                           0.94      5000
   macro avg       0.94      0.93      0.93      5000
weighted avg       0.94      0.94      0.94      5000

Confusion Matrix:
 [[ 779    0   59]
 [   0 2253  107]
 [  44  115 1643]]


## XGBoost Model

In [12]:
# encode target for XGBoost only
le = LabelEncoder()
risk_df["risk_score_encoded"] = le.fit_transform(risk_df["risk_score"])
print("Label Mapping: ",dict(zip(le.classes_,le.transform(le.classes_))))

Label Mapping:  {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}


In [13]:
# Time-based split for XGBoost
# Using same split_idx — consistent with LR and RF
risk_df = risk_df.sort_values("visit_date").reset_index(drop=True)
split_idx = int(len(risk_df) * 0.8)

risk_train_xgb = risk_df.iloc[:split_idx].copy()
risk_test_xgb  = risk_df.iloc[split_idx:].copy()

X_train_risk = risk_train_xgb[risk_features]
X_test_risk  = risk_test_xgb[risk_features]

# Separate XGB label variables — does NOT overwrite y_train_risk
y_train_risk_xgb = risk_train_xgb["risk_score_encoded"]
y_test_risk_xgb  = risk_test_xgb["risk_score_encoded"]

print("XGBoost train shape:", X_train_risk.shape)
print("XGBoost test shape :", X_test_risk.shape)

XGBoost train shape: (20000, 14)
XGBoost test shape : (5000, 14)


In [14]:
# XGBoost Model
risk_xgb_model = Pipeline(steps=[
    ("preprocessor", risk_preprocessor),
    ("classifier", XGBClassifier(
        objective="multi:softmax",
        num_class=3,
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    ))
])

# Train
risk_xgb_model.fit(X_train_risk, y_train_risk_xgb)

# Predict Train
risk_xgb_pred_train = risk_xgb_model.predict(X_train_risk)
# Predict Test
risk_xgb_pred_test  = risk_xgb_model.predict(X_test_risk)

# Save for comparison
xgb_acc = accuracy_score(y_test_risk_xgb, risk_xgb_pred_test)

print("Risk XGB Train Accuracy :", round(accuracy_score(y_train_risk_xgb, risk_xgb_pred_train), 4))
print("Risk XGB Test Accuracy  :", round(accuracy_score(y_test_risk_xgb,  risk_xgb_pred_test),  4))
print("Risk XGB Test Weighted F1:", round(f1_score(y_test_risk_xgb, risk_xgb_pred_test, average="weighted"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test_risk_xgb, risk_xgb_pred_test,
                             target_names=le.classes_))
print("Confusion Matrix:\n", confusion_matrix(y_test_risk_xgb, risk_xgb_pred_test))

Risk XGB Train Accuracy : 0.9941
Risk XGB Test Accuracy  : 0.955
Risk XGB Test Weighted F1: 0.9551

Classification Report:

              precision    recall  f1-score   support

        High       0.98      0.94      0.96       840
         Low       0.97      0.97      0.97      2360
      Medium       0.93      0.95      0.94      1800

    accuracy                           0.95      5000
   macro avg       0.96      0.95      0.95      5000
weighted avg       0.96      0.95      0.96      5000

Confusion Matrix:
 [[ 790    0   50]
 [   0 2278   82]
 [  16   77 1707]]


In [15]:
# Model Comparison
print("="*45)
print("Model A - RISK CLASSIFICATION COMPARISON")
print("="*45)
print(f"Logistic Regression: {baseline_acc:.4f}")
print(f"Random Forest      : {rf_acc:.4f}")
print(f"XGBoost            : {xgb_acc:.4f}")
print()
print(f"Best model Xgboost -> {max(baseline_acc, rf_acc, xgb_acc)}")

Model A - RISK CLASSIFICATION COMPARISON
Logistic Regression: 0.9208
Random Forest      : 0.9350
XGBoost            : 0.9550

Best model Xgboost -> 0.955
